<div style='background:#0b0e10;color:#f4f5f6;border:1px solid #2c3136;border-radius:18px;padding:28px 22px;text-align:center'><div style='font-family:monospace;font-size:12px;letter-spacing:.22em;color:#9aa1aa'>MINEIRO · PYTHON · v2.0.0a1</div><h1 style='font-size:36px;margin:10px 0 4px'>MINEIRO USERNAME INTELLIGENCE</h1><p style='font-size:17px;color:#a3a9b0;margin:0'>OSINT Investigation Workbench</p><p style='font-size:13px;color:#7f8790;margin-top:12px'>coleta → evidência → correlação → avaliação</p></div>

**Uso:** *Ambiente de execução → Executar tudo*. Tudo roda **dentro do notebook**: sem servidor, sem proxy e sem build.

> Use apenas identificadores públicos que você esteja autorizado a investigar. Um `FOUND` é uma observação, não prova de identidade.
>
> **Se algo falhar:** rode a célula **Diagnóstico** (2) e envie a saída. Ela mostra versões, rede, TLS e um scan de teste.

## 1 · Instalar
Instala o pacote `mineiro` (≈80 kB, depende só de `httpx`). Tenta a `main` e, se o PR do Python ainda não foi mergeado, usa a branch `feat/mineiro-py`.

In [ ]:
import importlib, subprocess, sys

REPO = 'https://github.com/Ridd1kulusC0d3r/mineirohml.git'
# 'main' vale depois do merge; 'feat/mineiro-py' vale enquanto o PR do Python estiver aberto.
BRANCHES = ['main', 'feat/mineiro-py']

for branch in BRANCHES:
    spec = f'mineiro @ git+{REPO}@{branch}#subdirectory=mineiro-py'
    done = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', spec], capture_output=True, text=True)
    if done.returncode == 0:
        print(f'pacote instalado da branch {branch}')
        break
else:
    raise RuntimeError('Não foi possível instalar o pacote:\n' + done.stderr[-1500:])

importlib.invalidate_caches()
import mineiro
print(f'Mineiro {mineiro.__version__} · Python {sys.version.split()[0]}')

## 2 · Diagnóstico
Confere Python, dependências, registry, DNS, TLS e faz 3 consultas reais. **Leva ~5 s.** Se qualquer coisa falhar depois, é aqui que aparece o motivo.

In [ ]:
from mineiro.doctor import doctor
_ = doctor()

## 3 · Escanear (formulário)
Preencha os campos à direita e execute. **quick** = 19 sites · **standard** = 49 com Evidence Engine e baseline · **full** = 984 em duas fases (demora). Não usa widgets: é o caminho mais robusto.

In [ ]:
#@title Escanear { display-mode: "form" }
alvo = "forgejo"  #@param {type:"string"}
preset = "standard"  #@param ["quick", "standard", "full"]
idioma = "pt"  #@param ["en", "pt", "es"]
requisito = "account_correlation"  #@param ["username_presence", "account_correlation", "digital_footprint", "developer_footprint", "threat_research_alias", "brand_impersonation"]

from mineiro.ui import run_form
assessment = await run_form(alvo, preset=preset, lang=idioma, requirement=requisito)

## 4 · Interface interativa (opcional)
Formulário com botões (usa `ipywidgets`). Se não aparecer ou não responder, use a célula 3: o resultado é o mesmo.

In [ ]:
from mineiro.ui import launch
ui = launch()

## 5 · Pela API Python (opcional)
O mesmo motor, sem widgets. No Colab dá para usar `await` direto numa célula.

In [ ]:
from mineiro import Scanner, build_assessment
from mineiro.export import write_report

async with Scanner(preset='quick') as scanner:
    results = await scanner.scan('forgejo')

assessment = build_assessment(results, 'forgejo', lang='pt')
for e in assessment['evidence']:
    if e['status'] == 'found':
        print(e['platform_name'], e['source_quality'], e['intelligence_priority_score'], e['url'])

path, manifest = write_report(assessment, results, 'mineiro-reports', 'html')
print('relatório:', path)

## 6 · Grafo, casos e diff (opcional)
Consulta local somente leitura (`MATCH`, `EDGE`, `SHARED`, `PATH`), export Cypher e Diff Intelligence entre duas coletas.

In [ ]:
from mineiro.graph import run_graph_query, graph_to_cypher, add_variant_candidates
from mineiro.variants import generate_username_variants
from mineiro.cases import CaseStore, diff_collections

graph = add_variant_candidates(assessment['graph'], 'forgejo', generate_username_variants('forgejo'))
print(run_graph_query('PATH from=TARGET to=PROFILE', graph)['summary'])
print(run_graph_query('MATCH candidate=true AND similarity>=90', graph)['summary'])

store = CaseStore('mineiro-reports/cases.sqlite')
case_id = store.get_or_create_case('forgejo')
store.add_snapshot(case_id, results, 'quick')
print(store.list_cases())

## 7 · Verificar integridade
Cada exportação gera um `.manifest.json` com o SHA-256 do arquivo. Isso prova integridade do conteúdo, não autoria nem cadeia de custódia.

In [ ]:
from mineiro.export import verify_manifest
print('íntegro:', verify_manifest(path))